In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.metrics import AUC

# # Import your modular code
# from src.model import build_resnet50_hybrid, combined_focal_dice_loss
# from src.dataset import create_tf_dataset
# from src.callbacks import F1ScoreCallback

2026-03-02 21:48:03.285169: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772488083.499167      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772488083.556968      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772488084.041074      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772488084.041129      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772488084.041132      55 computation_placer.cc:177] computation placer alr

In [2]:
# callback
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score, precision_score, recall_score

class F1ScoreCallback(tf.keras.callbacks.Callback):

    def __init__(self, val_data, y_val_true, verbose=1):
        super().__init__()
        self.val_data = val_data
        self.y_val_true = y_val_true
        self.best_f1 = 0
        self.best_thresh = 0.5
        self.verbose = verbose

    def on_epoch_end(self, epoch, logs=None):

        y_pred_probs = self.model.predict(self.val_data, verbose=0).flatten()

        best_f1 = 0
        best_thresh = 0.5

        for thresh in np.arange(0.1, 0.91, 0.01):
            preds = (y_pred_probs > thresh).astype(int)
            f1 = f1_score(self.y_val_true, preds)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = thresh

        self.best_f1 = best_f1
        self.best_thresh = best_thresh

        precision = precision_score(
            self.y_val_true,
            (y_pred_probs > best_thresh).astype(int)
        )

        recall = recall_score(
            self.y_val_true,
            (y_pred_probs > best_thresh).astype(int)
        )

        if self.verbose:
            print(
                f"\nEpoch {epoch+1}: "
                f"Best Thresh={best_thresh:.2f}, "
                f"F1={best_f1:.4f}, "
                f"Precision={precision:.4f}, "
                f"Recall={recall:.4f}"
            )

# Some built-in Keras callbacks used: Checkpoint, EarlyStopping, ReduceLR.

In [3]:
# dataset
import os
import numpy as np
import tensorflow as tf
import albumentations as A
import cv2

# Albumentations Pipeline
albumentations_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.GaussNoise(p=0.2),
    A.GaussianBlur(blur_limit=(1, 1), p=0.1),
])

# Load + Normalize + Resize
def load_npy_image(path, target_size=(128, 128), normalize=True):
    img = np.load(path).astype(np.float32)

    if normalize:
        img = (img - img.min()) / (img.max() - img.min() + 1e-5)

    img_resized = np.stack([
        cv2.resize(img[:, :, i], target_size, interpolation=cv2.INTER_LINEAR)
        for i in range(img.shape[-1])
    ], axis=-1)

    return img_resized.astype(np.float32)


def load_and_preprocess(image_path, label, augment=False):

    def _load(path):
        path = path.numpy().decode("utf-8")
        img = load_npy_image(path)

        if augment:
            img = albumentations_transform(image=img)["image"]

        return img

    image = tf.py_function(func=_load, inp=[image_path], Tout=tf.float32)
    image.set_shape([128, 128, 12])

    return image, tf.cast(label, tf.float32)

# tf.data Builder
def create_tf_dataset(
    df,
    folder_path,
    batch_size=16,
    augment=False,
    shuffle=True
):

    image_paths = [
        os.path.join(folder_path, f"{img_id}.npy")
        for img_id in df["ID"]
    ]

    labels = df["label"].values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)

    ds = ds.map(
        lambda path, label: load_and_preprocess(path, label, augment),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


In [4]:
# model
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.losses import BinaryFocalCrossentropy
import tensorflow.keras.backend as K

# Loss Functions
def dice_loss(y_true, y_pred):
    smooth = 1e-5
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (
        K.sum(y_true_f) + K.sum(y_pred_f) + smooth
    )


def combined_focal_dice_loss(y_true, y_pred):
    focal = BinaryFocalCrossentropy()(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return focal + dice

# Hybrid ResNet50 Model
def build_resnet50_hybrid(
    input_shape=(128, 128, 12),
    dropout_rate=0.4,
    trainable_resnet=False
):

    inputs = Input(shape=input_shape)

    # 12 → 3 channel projection block
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(3, (1, 1), padding="same", activation="relu")(x)

    # ResNet50 backbone
    resnet = ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(128, 128, 3)
    )

    resnet.trainable = trainable_resnet
    x = resnet(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs)

    return model, resnet

In [5]:
TRAIN_CSV = "/kaggle/input/datasets/varunsaxena16/landslide-detection-dataset/Train.csv"
TRAIN_DATA_DIR = "/kaggle/input/datasets/varunsaxena16/landslide-detection-dataset/train_data/train_data"

df = pd.read_csv(TRAIN_CSV)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))

Train samples: 5717
Validation samples: 1430


In [6]:
batch_size = 16

train_ds = create_tf_dataset(
    train_df,
    TRAIN_DATA_DIR,
    batch_size=batch_size,
    augment=True
)

val_ds = create_tf_dataset(
    val_df,
    TRAIN_DATA_DIR,
    batch_size=batch_size,
    augment=False
)

I0000 00:00:1772488113.382435      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772488113.388394      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [7]:
model, resnet_base = build_resnet50_hybrid()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        AUC(curve="ROC", name="auc"),
        AUC(curve="PR", name="auprc"),
    ]
)

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 12)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │         3,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 3)    │           195 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,733,348 (94.35 MB)

 Trainable params: 1,141,348 (4.35 MB)

 Non-trainable params: 23,592,000 (90.00 MB)

In [8]:
checkpoint_cb = ModelCheckpoint(
    "best_model.h5",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stop_cb = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_cb = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

f1_callback = F1ScoreCallback(
    val_ds,
    val_df["label"].values
)

In [9]:
print("Phase 1: Warm-up")

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=7,
    callbacks=[
        checkpoint_cb,
        early_stop_cb,
        reduce_lr_cb,
        f1_callback
    ]
)

Phase 1: Warm-up
Epoch 1/7


I0000 00:00:1772488125.339016     123 service.cc:152] XLA service 0x7d2574003500 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772488125.339077     123 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1772488125.339083     123 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1772488127.718544     123 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-02 21:48:51.770953: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-02 21:48:51.913322: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


  2/358 ━━━━━━━━━━━━━━━━━━━━ 31s 88ms/step - accuracy: 0.0469 - auc: 0.1724 - auprc: 0.0340 - loss: 1.1110               

I0000 00:00:1772488136.681876     123 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


357/358 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.7281 - auc: 0.5847 - auprc: 0.2189 - loss: 0.5434

...


Epoch 1: Best Thresh=0.13, F1=0.3048, Precision=0.1847, Recall=0.8725
358/358 ━━━━━━━━━━━━━━━━━━━━ 91s 200ms/step - accuracy: 0.7285 - auc: 0.5850 - auprc: 0.2191 - loss: 0.5430 - val_accuracy: 0.8245 - val_auc: 0.8075 - val_auprc: 0.3815 - val_loss: 0.4315 - learning_rate: 1.0000e-04
Epoch 2/7
357/358 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.8181 - auc: 0.7371 - auprc: 0.3885 - loss: 0.4235
Epoch 2: val_loss improved from 0.43151 to 0.36451, saving model to best_model.h5



Epoch 2: Best Thresh=0.10, F1=0.2283, Precision=0.1703, Recall=0.3466
358/358 ━━━━━━━━━━━━━━━━━━━━ 44s 124ms/step - accuracy: 0.8181 - auc: 0.7372 - auprc: 0.3886 - loss: 0.4233 - val_accuracy: 0.8336 - val_auc: 0.8545 - val_auprc: 0.4977 - val_loss: 0.3645 - learning_rate: 1.0000e-04
Epoch 3/7
357/358 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.8322 - auc: 0.7930 - auprc: 0.4605 - loss: 0.3848
Epoch 3: val_loss improved from 0.36451 to 0.32150, saving model to best_model.h5



Epoch 3: Best Thresh=0.10, F1=0.2251, Precision=0.1610, Recall=0.3745
358/358 ━━━━━━━━━━━━━━━━━━━━ 44s 122ms/step - accuracy: 0.8322 - auc: 0.7931 - auprc: 0.4605 - loss: 0.3847 - val_accuracy: 0.8392 - val_auc: 0.8804 - val_auprc: 0.5459 - val_loss: 0.3215 - learning_rate: 1.0000e-04
Epoch 4/7
357/358 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.8465 - auc: 0.8150 - auprc: 0.5053 - loss: 0.3659
Epoch 4: val_loss improved from 0.32150 to 0.31841, saving model to best_model.h5



Epoch 4: Best Thresh=0.10, F1=0.2546, Precision=0.1794, Recall=0.4382
358/358 ━━━━━━━━━━━━━━━━━━━━ 44s 122ms/step - accuracy: 0.8466 - auc: 0.8150 - auprc: 0.5054 - loss: 0.3659 - val_accuracy: 0.8601 - val_auc: 0.8856 - val_auprc: 0.6138 - val_loss: 0.3184 - learning_rate: 1.0000e-04
Epoch 5/7
357/358 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.8363 - auc: 0.8110 - auprc: 0.5030 - loss: 0.3781
Epoch 5: val_loss did not improve from 0.31841

Epoch 5: Best Thresh=0.10, F1=0.2167, Precision=0.1663, Recall=0.3108
358/358 ━━━━━━━━━━━━━━━━━━━━ 43s 121ms/step - accuracy: 0.8364 - auc: 0.8110 - auprc: 0.5031 - loss: 0.3780 - val_accuracy: 0.8517 - val_auc: 0.8941 - val_auprc: 0.6500 - val_loss: 0.3238 - learning_rate: 1.0000e-04
Epoch 6/7
357/358 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.8507 - auc: 0.8346 - auprc: 0.5646 - loss: 0.3544
Epoch 6: val_loss improved from 0.31841 to 0.30321, saving model to best_model.h5



Epoch 6: Best Thresh=0.10, F1=0.2095, Precision=0.1571, Recall=0.3147
358/358 ━━━━━━━━━━━━━━━━━━━━ 44s 122ms/step - accuracy: 0.8507 - auc: 0.8346 - auprc: 0.5646 - loss: 0.3544 - val_accuracy: 0.8601 - val_auc: 0.8986 - val_auprc: 0.6778 - val_loss: 0.3032 - learning_rate: 1.0000e-04
Epoch 7/7
357/358 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.8554 - auc: 0.8265 - auprc: 0.5489 - loss: 0.3485
Epoch 7: val_loss did not improve from 0.30321

Epoch 7: Best Thresh=0.10, F1=0.2385, Precision=0.1807, Recall=0.3506
358/358 ━━━━━━━━━━━━━━━━━━━━ 43s 120ms/step - accuracy: 0.8554 - auc: 0.8266 - auprc: 0.5489 - loss: 0.3485 - val_accuracy: 0.8538 - val_auc: 0.8838 - val_auprc: 0.6522 - val_loss: 0.3245 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 6.


In [10]:
print("Phase 2: Fine-tuning")

resnet_base.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=combined_focal_dice_loss,
    metrics=[
        "accuracy",
        AUC(curve="ROC", name="auc"),
        AUC(curve="PR", name="auprc"),
    ]
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[
        checkpoint_cb,
        early_stop_cb,
        reduce_lr_cb,
        f1_callback
    ]
)

Phase 2: Fine-tuning
Epoch 1/20
358/358 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - accuracy: 0.8015 - auc: 0.6836 - auprc: 0.3404 - loss: 1.2193
Epoch 1: val_loss did not improve from 0.30321

Epoch 1: Best Thresh=0.10, F1=0.0079, Precision=0.5000, Recall=0.0040
358/358 ━━━━━━━━━━━━━━━━━━━━ 148s 265ms/step - accuracy: 0.8015 - auc: 0.6838 - auprc: 0.3406 - loss: 1.2188 - val_accuracy: 0.8245 - val_auc: 0.5047 - val_auprc: 0.1878 - val_loss: 18.9012 - learning_rate: 1.0000e-05
Epoch 2/20
358/358 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.8147 - auc: 0.8242 - auprc: 0.5138 - loss: 0.8324
Epoch 2: val_loss did not improve from 0.30321

Epoch 2: Best Thresh=0.10, F1=0.1098, Precision=0.2338, Recall=0.0717
358/358 ━━━━━━━━━━━━━━━━━━━━ 58s 163ms/step - accuracy: 0.8147 - auc: 0.8242 - auprc: 0.5138 - loss: 0.8323 - val_accuracy: 0.8315 - val_auc: 0.7306 - val_auprc: 0.4919 - val_loss: 3.4825 - learning_rate: 1.0000e-05
Epoch 3/20
358/358 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.8226 

In [11]:
y_true = val_df["label"].values
y_prob = model.predict(val_ds).flatten()

threshold = 0.5
y_pred = (y_prob > threshold).astype(int)

print(classification_report(y_true, y_pred))

90/90 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step
              precision    recall  f1-score   support

           0       0.83      0.82      0.83      1179
           1       0.20      0.22      0.21       251

    accuracy                           0.71      1430
   macro avg       0.52      0.52      0.52      1430
weighted avg       0.72      0.71      0.72      1430

